In [36]:
import re 
import pandas as pd

In [48]:
f = open ('WhatsApp Chat with MCA AU 2023 UNOFFICIAL.txt','r',encoding='utf-8')

In [50]:
data = f.read()

In [52]:
print(data)

4/30/24, 1:30 PM - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
11/17/23, 5:12 PM - ~ Reddy created group "MCA AU 2023 UNOFFICIAL"
4/30/24, 1:30 PM - You joined using this group's invite link
4/30/24, 2:09 PM - Aark Kadiyam Mca Aurora joined using this group's invite link
4/30/24, 2:10 PM - Kusuma Darshi Mca Aurora: <Media omitted>
4/30/24, 3:04 PM - +91 89772 58957: <Media omitted>
4/30/24, 6:15 PM - Magnet 😍 joined using this group's invite link
5/1/24, 9:56 AM - Kusuma Darshi Mca Aurora: Proposed Syllabus 
Module 1: Introduction to Data Science  
Basic Terminology, Data science Venn diagram, Types of Data, The Four Levels of Data. 
Module 2: Importance of Data Science 
Need for Data Science, Five steps of Data Science, Business Intelligence and Process, 
Prerequisites for Data Scientist, Components of Data Science, Tools and Skills needed 
Module 3: Data Science Process 
Introduction, Prio

In [54]:
pattern = '\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s'

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\vinod\AppData\Local\Temp\ipykernel_11928\685747602.py:1: SyntaxWarning: invalid escape sequence '\d'
  pattern = '\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s'


In [70]:
messages= re.split(pattern, data)
messages

['4/30/24, 1:30\u202fPM - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.\n11/17/23, 5:12\u202fPM - ~\u202fReddy created group "MCA AU 2023 UNOFFICIAL"\n4/30/24, 1:30\u202fPM - You joined using this group\'s invite link\n4/30/24, 2:09\u202fPM - Aark Kadiyam Mca Aurora joined using this group\'s invite link\n4/30/24, 2:10\u202fPM - Kusuma Darshi Mca Aurora: <Media omitted>\n4/30/24, 3:04\u202fPM - +91 89772 58957: <Media omitted>\n4/30/24, 6:15\u202fPM - Magnet 😍 joined using this group\'s invite link\n5/1/24, 9:56\u202fAM - Kusuma Darshi Mca Aurora: Proposed Syllabus \nModule 1: Introduction to Data Science  \nBasic Terminology, Data science Venn diagram, Types of Data, The Four Levels of Data. \nModule 2: Importance of Data Science \nNeed for Data Science, Five steps of Data Science, Business Intelligence and Process, \nPrerequisites for Data Scientist, Components of Data Science, Tools and Ski

In [72]:
dates = re.findall(pattern, data)
dates

[]

In [76]:
import re
import pandas as pd

def preprocess(data):
    pattern = '\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s'

    messages = re.split(pattern, data)[1:]
    dates = re.findall(pattern, data)

    df = pd.DataFrame({'user_message': messages, 'message_date': dates})
    # convert message_date type
    df['message_date'] = pd.to_datetime(df['message_date'], format='%d/%m/%Y, %H:%M - ')

    df.rename(columns={'message_date': 'date'}, inplace=True)

    users = []
    messages = []
    for message in df['user_message']:
        entry = re.split('([\w\W]+?):\s', message)
        if entry[1:]:  # user name
            users.append(entry[1])
            messages.append(" ".join(entry[2:]))
        else:
            users.append('group_notification')
            messages.append(entry[0])

    df['user'] = users
    df['message'] = messages
    df.drop(columns=['user_message'], inplace=True)

    df['only_date'] = df['date'].dt.date
    df['year'] = df['date'].dt.year
    df['month_num'] = df['date'].dt.month
    df['month'] = df['date'].dt.month_name()
    df['day'] = df['date'].dt.day
    df['day_name'] = df['date'].dt.day_name()
    df['hour'] = df['date'].dt.hour
    df['minute'] = df['date'].dt.minute

    period = []
    for hour in df[['day_name', 'hour']]['hour']:
        if hour == 23:
            period.append(str(hour) + "-" + str('00'))
        elif hour == 0:
            period.append(str('00') + "-" + str(hour + 1))
        else:
            period.append(str(hour) + "-" + str(hour + 1))

    df['period'] = period

    return df

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\w'
C:\Users\vinod\AppData\Local\Temp\ipykernel_11928\1533445789.py:5: SyntaxWarning: invalid escape sequence '\d'
  pattern = '\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s'
C:\Users\vinod\AppData\Local\Temp\ipykernel_11928\1533445789.py:19: SyntaxWarning: invalid escape sequence '\w'
  entry = re.split('([\w\W]+?):\s', message)


In [84]:
import re
import pandas as pd

def preprocess(data):
    # Use raw string (r"...") to avoid escape sequence warnings
    pattern = r'\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s'

    # Split messages and extract timestamps
    messages = re.split(pattern, data)
    dates = re.findall(pattern, data)

    # Ensure data isn't empty to avoid index errors
    if len(messages) > 1:
        messages = messages[1:]  # Exclude first empty entry
    else:
        return pd.DataFrame()  # Return empty DataFrame if no messages

    # Create DataFrame
    df = pd.DataFrame({'user_message': messages, 'message_date': dates})

    # Convert 'message_date' column to datetime format with error handling
    df['message_date'] = pd.to_datetime(df['message_date'], format='%d/%m/%Y, %H:%M - ', errors='coerce')
    df.rename(columns={'message_date': 'date'}, inplace=True)

    # Extract users and messages
    users = []
    message_texts = []

    for message in df['user_message']:
        entry = re.split(r'([\w\W]+?):\s', message, maxsplit=1)  # Use raw string
        if len(entry) > 1:  # If user message contains a sender
            users.append(entry[1].strip())
            message_texts.append(entry[2].strip())
        else:  # System-generated messages (e.g., "You created this group")
            users.append('group_notification')
            message_texts.append(entry[0].strip())

    df['user'] = users
    df['message'] = message_texts
    df.drop(columns=['user_message'], inplace=True)

    # Extract additional datetime attributes
    df['only_date'] = df['date'].dt.date
    df['year'] = df['date'].dt.year
    df['month_num'] = df['date'].dt.month
    df['month'] = df['date'].dt.month_name()
    df['day'] = df['date'].dt.day
    df['day_name'] = df['date'].dt.day_name()
    df['hour'] = df['date'].dt.hour
    df['minute'] = df['date'].dt.minute

    # Create 'period' column
    df['period'] = df['hour'].apply(lambda hour: f"{hour}-00" if hour == 23 else (f"00-{hour + 1}" if hour == 0 else f"{hour}-{hour + 1}"))

    return df


In [98]:
data['dates']=pd.to_datetime(df['dates'], format='%d/%m/%Y, %H:%M - ')



NameError: name 'df' is not defined